## Dependencies

In [1]:
import pathlib
import numpy as np
import pandas as pd

## Download & Create EC -> GO Term Mapping

In [2]:
!curl -s -L -o ec2go.txt http://www.geneontology.org/external2go/ec2go

with open('ec2go.txt', 'r') as file:
    data = file.readlines()

# A dictionary to store InterPro codes and their associated GO terms
# If there are multiple GO terms for each interpro entry, they will be stored in an array
ec2go = {}

# Extract InterPro code and GO code for a given line
def extract_codes(line):
    parts = line.strip().split(' ')
    ec_code = parts[0].split(':')[1]
    go_code = parts[-1]
    return ec_code, go_code

for line in data:
    if line[0] == '!':
        continue
    ec_code, go_code = extract_codes(line)
   
    if ec_code in ec2go:
        ec2go[ec_code].append(go_code)
    else:
        ec2go[ec_code] = [go_code]

## Download & Create dataset

In [3]:
flat = "https://www.ebi.ac.uk/thornton-srv/m-csa/media/flat_files/curated_data.csv"
curated_df = pd.read_csv("downloads/curated_data.csv", on_bad_lines="skip")

In [9]:
curated_df

,M-CSA ID,Uniprot IDs,PDB,EC,residue/reactant/product/cofactor,PDB code,chain/kegg compound,resid/chebi id,function location/name,role,role type,role group
0,M0001,P56868,1b73,5.1.1.3,residue,Asp,A,7,side_chain,activator,spectator,activator
1,M0001,P56868,1b73,5.1.1.3,residue,Asp,A,7,side_chain,hydrogen bond acceptor,interaction,NaN
2,M0001,P56868,1b73,5.1.1.3,residue,Asp,A,7,side_chain,proton acceptor,reactant,proton shuttle (general acid/base)
3,M0001,P56868,1b73,5.1.1.3,residue,Asp,A,7,side_chain,electrostatic stabiliser,spectator,electrostatic interaction
4,M0001,P56868,1b73,5.1.1.3,residue,Asp,A,7,side_chain,hydrogen bond donor,interaction,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
23760,M0966,P15723,4x9e,3.1.5.1,residue,His,A,117,side_chain,metal ligand,interaction,NaN
23761,M0966,P15723,4x9e,3.1.5.1,residue,Asp,A,118,side_chain,metal ligand,interaction,NaN
23762,M0966,P15723,4x9e,3.1.5.1,residue,Asp,A,268,side_chain,metal ligand,interaction,NaN
23763,M0966,P15723,4x9e,3.1.5.1,residue,Arg,B,442,side_chain,unknown,spectator,NaN


In [10]:
def GO_mapping(dataframe, ec2go=ec2go):
    output_go = []
    output_nogo = []
    proteins = np.unique(dataframe["Uniprot IDs"].values)

    for protein in proteins:
        new_df = dataframe[dataframe["Uniprot IDs"] == protein].copy()

        indices = np.unique(new_df["resid/chebi id"].values) 
        indices = indices.astype(int).tolist()
        
        
        enzyme_class = new_df["EC"].values[0]

        try:
            go_term = ec2go[enzyme_class]
            output_go.append((protein, indices, enzyme_class, go_term))
            # print(f"Protein: {protein}, Indices: {indices}, Enzyme Class: {enzyme_class}, GO Term: {go_term}")
        except KeyError:
            output_nogo.append((protein, indices, enzyme_class))
            # print(f"Protein: {protein}, Indices: {indices}, Enzyme Class: {enzyme_class}")
            
    return output_go, output_nogo

dataset, _ = GO_mapping(curated_df)   

In [11]:
import pandas as pd
dataset_df = pd.DataFrame(dataset, columns=['UniprotID', 'AnnotatedIndices', 'EnzymeClass', 'GOTerm'])
# dataset_df.to_csv('datasets/csa_annot.csv', index=False)

In [12]:
dataset_df

,UniprotID,AnnotatedIndices,EnzymeClass,GOTerm
0,A0A0F6FBL7,"[14, 83, 197]",5.1.1.13,[GO:0047689]
1,A0QTN8,"[162, 191, 217, 242, 266, 294, 18420]",5.5.1.1,[GO:0018849]
2,A2RJT9,"[43, 130, 57618]",1.3.98.1,[GO:1990663]
3,A4XF23,"[147, 159, 210, 212, 236, 262, 283, 339, 402, ...",4.2.1.8,[GO:0008927]
4,A5JTM5,"[64, 86, 90, 114, 137, 145]",3.8.1.7,[GO:0018787]
...,...,...,...,...
841,Q9ZAG3,"[53, 55, 99, 101, 132]",3.3.2.8,[GO:0018744]
842,Q9ZF13,"[50, 127, 128, 196, 198, 225, 254]",3.2.1.78,[GO:0016985]
843,Q9ZHG9,"[114, 197, 199, 200, 223, 360, 597326]",4.4.1.35,[GO:0044540]
844,Q9ZHI0,"[114, 116, 118, 221, 312, 406, 409, 422, 427]",6.5.1.2,[GO:0003911]


## Get sequence and do features/labels for given go term

In [13]:
from utils import fetch_sequence_from_redundant
dataset_df['Sequence'] = dataset_df['UniprotID'].apply(fetch_sequence_from_redundant)
dataset_df['Sequence'] = [seq for _, (_, seq) in dataset_df['Sequence'].items()]

KeyboardInterrupt: 

In [ ]:
# import pandas as pd
# dataset_df = pd.read_csv('datasets/csa_annot.csv', sep='\t').dropna()
dataset_df = dataset_df[[len(t) < 850 for t in dataset_df['Sequence']]]
dataset_df.to_csv('datasets/csa_annot.csv', index=False, sep='\t')

In [39]:
dataset_df.to_csv('datasets/csa_annot.csv', index=False, sep='\t')
dataset_df.tail()

,UniprotID,AnnotatedIndices,EnzymeClass,GOTerm,Sequence
841,Q9ZAG3,"[53, 55, 99, 101, 132]",3.3.2.8,[GO:0018744],MTSKIEQPRWASKDSAAGAASTPDEKIVLEFMDALTSNDAAKLIEY...
842,Q9ZF13,"[50, 127, 128, 196, 198, 225, 254]",3.2.1.78,[GO:0016985],GLHVKNGRLYEANGQEFIIRGVSHPHNWYPQHTQAFADIKSHGANT...
843,Q9ZHG9,"[114, 197, 199, 200, 223, 360, 597326]",4.4.1.35,[GO:0044540],MADPVNLIPDRHQFPGLANKTYFNFGGQGILPTVALEAITAMYGYL...
844,Q9ZHI0,"[114, 116, 118, 221, 312, 406, 409, 422, 427]",6.5.1.2,[GO:0003911],MTREEARRRINELRDLIRYHNYRYYVLADPEISDAEYDRLLRELKE...
845,Q9ZP19,"[88, 109, 118, 240, 244, 261, 274, 47292]",1.10.3.1,[GO:0004097],APIQAPEISKCVVPPADLPPGAVVDNCCPPVASNIVDYKLPAVTTM...
